In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

model.eval()
print("model is on:", next(model.parameters()).device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

model is on: cuda:0


In [ ]:
from datasets import load_dataset

test_data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split = "test")

In [ ]:
def evaluate_perplexity(model, dataset, tokenizer, device, max_len = 1024, stride = 512):

  text = "\n\n".join(dataset["text"])
  encodings = tokenizer(text, return_tensors = "pt")
  input_ids_all = encodings.input_ids.to(device)
  seq_len = input_ids_all.size(1)

  prev_end = 0
  nll_sum = 0
  n_tokens = 0

  for begin in range(0, seq_len, stride):
    end = min(begin + max_len, seq_len)
    target_len = end - prev_end

    input_ids = input_ids_all[:, begin:end]
    target_ids = input_ids.clone()
    target_ids[:, :-target_len] = -100

    with torch.no_grad():
      output = model(input_ids, labels = target_ids)
      neg_log_loss = output.loss

    nll_sum += neg_log_loss * target_len
    n_tokens += target_len
    prev_end = end
    if end == seq_len:
      break

  avg_nll = nll_sum / n_tokens
  perplexity = torch.exp(avg_nll)
  return perplexity.item()

In [ ]:
baseline_ppl = evaluate_perplexity(model, test_data, tokenizer, device)
print("The Baseline perplexity for FP32 Precision: ", baseline_ppl)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (287644 > 1024). Running this sequence through the model will result in indexing errors


The Baseline perplexity for FP32 Precision:  39.25837707519531


In [ ]:
def analyse_model_size(model):
  quantize_bytes = 0
  total_bytes = 0
  skip_bytes = 0

  targets = ("c_attn", "c_proj", "c_fc")

  for name, param in model.named_parameters():
    param_bytes = param.numel() * param.element_size()
    total_bytes += param_bytes

    is_weight = name.endswith(".weight")
    is_target = any(t in name for t in targets)

    if is_weight and is_target:
      quantize_bytes += param_bytes
      bucket = "QUANTIZE"
    else:
      skip_bytes += param_bytes
      bucket = "skip"

    mb = param_bytes / 1e6
    print(f"{bucket:9} {mb:6.2f}MB  {tuple(param.shape)!s:18} {name}")

  return total_bytes, quantize_bytes, skip_bytes

total, quant, skip = analyse_model_size(model)
print(f"\nTotal model size:      {total/1e6:7.2f} MB")
print(f"Weights we quantize:   {quant/1e6:7.2f} MB  ({100*quant/total:.1f}% of total)")
print(f"Weights we skip:       {skip/1e6:7.2f} MB  ({100*skip/total:.1f}% of total)")

skip      154.39MB  (50257, 768)       transformer.wte.weight
skip        3.15MB  (1024, 768)        transformer.wpe.weight
skip        0.00MB  (768,)             transformer.h.0.ln_1.weight
skip        0.00MB  (768,)             transformer.h.0.ln_1.bias
QUANTIZE    7.08MB  (768, 2304)        transformer.h.0.attn.c_attn.weight
skip        0.01MB  (2304,)            transformer.h.0.attn.c_attn.bias
QUANTIZE    2.36MB  (768, 768)         transformer.h.0.attn.c_proj.weight
skip        0.00MB  (768,)             transformer.h.0.attn.c_proj.bias
skip        0.00MB  (768,)             transformer.h.0.ln_2.weight
skip        0.00MB  (768,)             transformer.h.0.ln_2.bias
QUANTIZE    9.44MB  (768, 3072)        transformer.h.0.mlp.c_fc.weight
skip        0.01MB  (3072,)            transformer.h.0.mlp.c_fc.bias
QUANTIZE    9.44MB  (3072, 768)        transformer.h.0.mlp.c_proj.weight
skip        0.00MB  (768,)             transformer.h.0.mlp.c_proj.bias
skip        0.00MB  (768,)          